In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "raster_mosaicking"))
from raster_mosaic import process_folders
from pathlib import Path

In [ ]:
#Define the folders containing rasters to mosaic, and the shapefiles for clipping
folders = [
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\cocoa_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\coffee_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\palm_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\rubber_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\maize_2020"),
       # Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\rice_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\timber_2020"),
]
aoi_paths = [
    Path(r"C:\Users\AFahrezi\OneDrive - CIFOR-ICRAF\FAO\AOI\FAO_RCP51_AOI.shp"),
]
output_root = Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\mosaicked_all_countries\mosaic_2020_data")
#Run the full workflow automatically
results = process_folders(
    folders=folders,
    aoi_paths=aoi_paths,
    output_root=output_root,
    method="max",
    nodata=0,
    compress="lzw",
)

print(f"Created {len(results)} output rasters")
for path in results:
    print(path)


ValueError: array is too big; `arr.size * arr.dtype.itemsize` is larger than the maximum possible size.

In [ ]:
khm_folder = [Path(r"C:\Data Spasial\Rice_GDP\RAWDATA\2023_rice\KHM")]
mmr_folder = [Path(r"C:\Data Spasial\Rice_GDP\RAWDATA\2023_rice\MMR")]
phl_folder = [Path(r"C:\Data Spasial\Rice_GDP\RAWDATA\2023_rice\PHL")]
tha_folder = [Path(r"C:\Data Spasial\Rice_GDP\RAWDATA\2023_rice\THA")]
vnm_folder = [Path(r"C:\Data Spasial\Rice_GDP\RAWDATA\2023_rice\VNM")]
khm_aoi = [Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\CAMBODIA\CAMBODIA_AOI.shp")]
mmr_aoi = [Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\MYANMAR\MYANMAR_AOI.shp")]
phl_aoi = [Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\PHILIPHINE\PHILIPINE_AOI.shp")]
tha_aoi = [Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\THAILAND\Thai_AOI.shp")]
vnm_aoi = [Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\VIETNAM\Vietnam_AOI.shp")]
output_rice = Path(r"C:\Data Spasial\Rice_GDP\FINAL")


In [ ]:
mosaic_rice_khm = process_folders(folders=khm_folder,aoi_paths=khm_aoi,
    output_root=output_rice,
    method="max",
    nodata=0,
    compress="lzw")
mosaic_rice_mmr = process_folders(aoi_paths=mmr_aoi,folders=mmr_folder,
                                  output_root=output_rice,method="max",nodata=0,compress="lzw")
mosaic_rice_phl = process_folders(aoi_paths=phl_aoi,folders=phl_folder,
                                  output_root=output_rice,method="max",nodata=0,compress="lzw")
mosaic_rice_tha = process_folders(aoi_paths=tha_aoi,folders=tha_folder,
                                  output_root=output_rice,method="max",nodata=0,compress="lzw")
mosaic_rice_vnm = process_folders(aoi_paths=vnm_aoi,folders=vnm_folder,
                                  output_root=output_rice,method="max",nodata=0,compress="lzw")

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from pathlib import Path

# 1. Define your paths
raster_dir = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\rasters\2020_data")
# Now pointing to a single master shapefile instead of a directory
master_shapefile_path = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\AOI\FAO_RCP51_Country.shp") 
output_dir = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\clipped_output")

output_dir.mkdir(parents=True, exist_ok=True)

# 2. Load the master shapefile ONCE (saves time and memory)
print("Loading master shapefile...")
sea_gdf = gpd.read_file(master_shapefile_path)

# 3. Iterate through all raster files in the directory
for raster_path in raster_dir.glob("*.tif"): 
    
    # Extract country code from filename: palm_binary_KHM_2024 -> KHM
    # name.split('_') = ['palm', 'binary', 'KHM', '2024']
    filename_parts = raster_path.stem.split('_')
    
    # Safety check: ensure the filename has enough parts
    if len(filename_parts) < 3:
        print(f"Skipping {raster_path.name}: Filename format unrecognized.")
        continue
        
    country_code = filename_parts[2] 
    
    print(f"Processing {country_code}...")

    # 4. Filter the master shapefile for the specific country
    # This selects only the row(s) where the ISO_A3 column matches our country code
    country_gdf = sea_gdf[sea_gdf['ISO_A3'] == country_code]
    
    if country_gdf.empty:
        print(f"  Warning: No features found for {country_code} in the shapefile. Skipping...")
        continue

    # 5. Open the raster file
    with rasterio.open(raster_path) as src:
        
        # 6. Ensure CRS match for the specific country slice
        if country_gdf.crs != src.crs:
            print(f"  Reprojecting {country_code} boundary to match raster CRS...")
            country_gdf = country_gdf.to_crs(src.crs)
            
        # Extract the geometries as a list for rasterio
        geometries = country_gdf.geometry.tolist()
        
        try:
            # 7. Apply the mask
            out_image, out_transform = mask(src, geometries, crop=True)
            
            # 8. Update metadata
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "compress": "lzw"
            })
            
            # 9. Save the clipped raster
            out_file = output_dir / f"{raster_path.stem}_clipped.tif"
            with rasterio.open(out_file, "w", **out_meta) as dest:
                dest.write(out_image)
                
            print(f"  Successfully saved: {out_file.name}")
            
        except ValueError as e:
            print(f"  Error masking {country_code} ({raster_path.name}): {e}")

print("Batch clipping complete.")

In [2]:
#mosaic all rasters
created = process_folders(
    folders=[
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\cocoa_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\coffee_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\palm_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\rubber_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\maize_2020"),
       # Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\rice_2020"),
        Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\timber_2020"),
    ],
    aoi_paths= [Path(r"C:\Users\AFahrezi\OneDrive - CIFOR-ICRAF\FAO\AOI\FAO_RCP51_AOI.shp")],
    output_root=Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\mosaicked_all_countries\mosaic_2020_data"),
    method="max",
    nodata=0,
    compress="lzw",
)

ValueError: array is too big; `arr.size * arr.dtype.itemsize` is larger than the maximum possible size.

tesy